**Import Libraries**

In [ ]:
from pypdf import PdfReader

import re
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

**Read PDF**

In [5]:
reader = PdfReader('E:\Machine Learning\Projects\PDF Q&A Assistant\data\Stanford ML Notes.pdf')

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Abid Kazmi\AppData\Local\Temp\ipykernel_43652\2327510883.py:1: SyntaxWarning: invalid escape sequence '\M'
  reader = PdfReader('E:\Machine Learning\Projects\PDF Q&A Assistant\data\Stanford ML Notes.pdf')


In [6]:
pdf = ""

In [7]:
for page in reader.pages:
    pdf += page.extract_text()

In [8]:
pdf

'CS229 Lecture Notes\nAndrew Ng and Tengyu Ma\nJune 11, 2023Contents\nI Supervised learning 5\n1 Linear regression 8\n1.1 LMS algorithm . . . . . . . . . . . . . . . . . . . . . . . . . . 9\n1.2 The normal equations . . . . . . . . . . . . . . . . . . . . . . . 13\n1.2.1 Matrix derivatives . . . . . . . . . . . . . . . . . . . . . 13\n1.2.2 Least squares revisited . . . . . . . . . . . . . . . . . . 14\n1.3 Probabilistic interpretation . . . . . . . . . . . . . . . . . . . . 15\n1.4 Locally weighted linear regression (optional reading) . . . . . . 17\n2 Classiﬁcation and logistic regression 20\n2.1 Logistic regression . . . . . . . . . . . . . . . . . . . . . . . . 20\n2.2 Digression: the perceptron learning algorithm . . . . . . . . . 23\n2.3 Multi-class classiﬁcation . . . . . . . . . . . . . . . . . . . . . 24\n2.4 Another algorithm for maximizing ℓ(θ) . . . . . . . . . . . . . 27\n3 Generalized linear models 29\n3.1 The exponential family . . . . . . . . . . . . . . . . . . . . . .

**Define and apply preprocess & create_chunks function**

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
# Preprocessing function before using sentence embedder
def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

In [ ]:
# Creates chunks of chunk_size words from pdf
def create_chunks(words, chunk_size = 200, overlap = 50):
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += (chunk_size-overlap)

    return chunks

We divide the cleaned pdf text into chunks of 200 words. The number of chunks will correspond to the number
of vectors in the multi-dimensional space.

In [12]:
words = pdf.split()

In [13]:
chunks = create_chunks(words)

In [14]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Chunk 1: CS229 Lecture Notes Andrew Ng and Tengyu Ma June 11, 2023Contents I Supervised learning 5 1 Linear regression 8 1.1 LMS algorithm . . . . . . . . . . . . . . . . . . . . . . . . . . 9 1.2 The normal equations . . . . . . . . . . . . . . . . . . . . . . . 13 1.2.1 Matrix derivatives . . . . . . . . . . . . . . . . . . . . . 13 1.2.2 Least squares revisited . . . . . . . . . . . . . . . . . . 14 1.3 Probabilistic interpretation . . . . . . . . . . . . . . . . . . . . 15 1.4 Locally weighted linear regression (optional reading) . . . . . . 17 2 Classiﬁcation and logistic regression 20 2.1 Logistic regression . . . . . . . . . . . . . . . . . . . . . . . . 20 2.2 Digression: the
Chunk 2: Locally weighted linear regression (optional reading) . . . . . . 17 2 Classiﬁcation and logistic regression 20 2.1 Logistic regression . . . . . . . . . . . . . . . . . . . . . . . . 20 2.2 Digression: the perceptron learning algorithm . . . . . . . . . 23 2.3 Multi-class classiﬁcation . . . . . 

In [15]:
cleaned_chunks = [preprocess(chunk) for chunk in chunks]

In [16]:
for i, chunk in enumerate(cleaned_chunks):
    print(f"Chunk {i+1}: {chunk}")

Chunk 1: cs229 lecture note andrew ng tengyu june 11 2023contents supervised learning 5 1 linear regression 8 1 1 lm algorithm 9 1 2 normal equation 13 1 2 1 matrix derivative 13 1 2 2 least square revisited 14 1 3 probabilistic interpretation 15 1 4 locally weighted linear regression optional reading 17 2 classi cation logistic regression 20 2 1 logistic regression 20 2 2 digression
Chunk 2: locally weighted linear regression optional reading 17 2 classi cation logistic regression 20 2 1 logistic regression 20 2 2 digression perceptron learning algorithm 23 2 3 multi class classi cation 24 2 4 another algorithm maximizing 27 3 generalized linear model 29 3 1 exponential family 29 3 2 constructing glms 31 3 2 1 ordinary least square 32 3 2 2 logistic regression
Chunk 3: 31 3 2 1 ordinary least square 32 3 2 2 logistic regression 33 4 generative learning algorithm 34 4 1 gaussian discriminant analysis 35 4 1 1 multivariate normal distribution 35 4 1 2 gaussian discriminant analysis mode

**Apply Sentence Transformer**

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

e:\Machine Learning\Projects\PDF Q&A Assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abid Kazmi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3928.36it/s]


In [17]:
chunk_embeddings = model.encode(chunks)

In [18]:
chunk_embeddings.shape

(467, 384)

**Test with query**

In [1]:
query = "Markov decision process"

In [2]:
cleaned_query = preprocess(query)

NameError: name 'preprocess' is not defined

In [ ]:
cleaned_query

'markov decision process'

In [ ]:
query_embedding = model.encode([query])

In [ ]:
query_embedding.shape

(1, 384)

Use cosine similarity to find top-k most similar results to our query

In [ ]:
similarities = cosine_similarity(
    query_embedding,
    chunk_embeddings
)[0]

In [ ]:
similarities

array([ 1.85035765e-01,  1.26260713e-01,  2.35087141e-01,  2.02039778e-01,
        2.09692091e-01,  1.58566535e-01,  1.05204239e-01,  8.85184109e-02,
        1.16781771e-01,  1.25077382e-01,  2.11100042e-01,  3.87864769e-01,
        2.95594335e-01,  1.94715291e-01,  2.36786351e-01,  1.85697585e-01,
        2.24312842e-01,  1.31139100e-01,  2.04773858e-01,  2.54368246e-01,
        2.18921125e-01,  2.01642826e-01,  1.42670035e-01,  1.69825792e-01,
        2.06874698e-01,  1.94995522e-01,  1.77226245e-01,  1.91795737e-01,
        2.05779582e-01,  1.46327376e-01,  3.89661081e-03,  1.43642351e-01,
        2.29917243e-01,  1.40047371e-01,  6.91362843e-03,  6.45912588e-02,
        1.47315204e-01,  1.64259553e-01,  1.99331909e-01,  1.63752839e-01,
        2.04517633e-01,  2.13665739e-01,  2.07612097e-01,  1.38293445e-01,
        1.52145386e-01,  2.74467945e-01,  2.21095756e-01,  1.84452981e-01,
        1.96386874e-01,  2.49494642e-01,  2.93619514e-01,  3.17769349e-01,
        2.51171768e-01,  

In [ ]:
top_k = 5

top_indices = np.argsort(similarities)[-top_k:][::-1]

In [ ]:
top_chunks = [chunks[i] for i in top_indices]

In [ ]:
print("\nTOP CHUNKS:\n")
for c in top_chunks:
    print("-", c, "\n")


TOP CHUNKS:

- negative rewards for either moving backwards or falling over. It will then be the learning algorithm’s job to ﬁgure out how to choose actions over time so as to obtain large rewards. Reinforcement learning has been successful in applications as diverse as autonomous helicopter ﬂight, robot legged locomotion, cell-phone network routing, marketing strategy selection, factory control, and eﬃcient web-page indexing. Our study of reinforcement learning will begin with a deﬁnition of the Markov decision processes (MDP), which provides the formalism in which RL problems are usually posed. 187188 15.1 Markov decision processes A Markov decision process is a tuple ( S, A, {Psa}, γ, R), where: • S is a set of states. (For example, in autonomous helicopter ﬂight, S might be the set of all possible positions and orientations of the heli- copter.) • A is a set of actions. (For example, the set of all possible directions in which you can push the helicopter’s control sticks.) • Psa a

**Retrieve most similar sentences from top-k chunks**

In [ ]:
# Cleaning function applied to fix punctuation issues before using sent_tokenize
def sentence_clean(text):
    text = re.sub(r'([.?!])([A-Za-z])', r'\1 \2', text)
    return text.strip()

In [ ]:
# This block is responsible for extracting the index of a sentence within
# the top chunks. We will later use this index to display the preceeding
# and succeeding sentences for context.
sentence_data = []

for c_idx, chunk in enumerate(top_chunks):
    sentences = sent_tokenize(chunk)

    for s_idx, s in enumerate(sentences):
        if len(s.split()) > 5:
            sentence_data.append({
                "chunk_id": c_idx,
                "sentences": sentences,
                "sent_idx": s_idx,
                "text": s
            })

There will be a seperate vector space for sentences

In [ ]:
sentence_texts = [x["text"] for x in sentence_data]

sentence_embeddings = model.encode(
    sentence_texts,
    normalize_embeddings = True
)

In [ ]:
query_embeddings = model.encode([query], normalize_embeddings=True)

sentence_similarities = cosine_similarity(
    query_embedding,
    sentence_embeddings
)[0]

In [ ]:
sentence_similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

In [ ]:
sentence_similarities

array([0.18888764, 0.38044372, 0.31880212, 0.68593365, 0.650049  ,
       0.14968841, 0.24501322, 0.12174039, 0.4449477 , 0.25987267,
       0.10121028, 0.38469276, 0.47834712, 0.33155185, 0.47466922,
       0.33868015, 0.2892496 , 0.2838382 , 0.3631474 , 0.22787707,
       0.3362404 , 0.5214014 , 0.5323448 , 0.28308558, 0.5462755 ,
       0.3816693 , 0.14164153, 0.12027298, 0.33480692, 0.5835527 ,
       0.41211584, 0.37522185, 0.21743327, 0.35102993, 0.26043022,
       0.21258274, 0.20062216, 0.3098877 , 0.21421778, 0.31387964,
       0.03734066, 0.07702802, 0.07878913, 0.5007566 ], dtype=float32)

In [ ]:
top_indices = np.argsort(sentence_similarities)[-3:][::-1]

In [ ]:
expanded_results = []

for idx in top_indices:
    item = sentence_data[idx]

    sentences = item["sentences"]
    sent_idx = item["sent_idx"]

    prev_sent = sentences[sent_idx - 1] if sent_idx - 1 >= 0 else ""
    curr_sent = sentences[sent_idx]
    next_sent = sentences[sent_idx + 1] if sent_idx + 1 < len(sentences) else ""

    expanded_results.append({
        "previous": prev_sent,
        "sentence": curr_sent,
        "next": next_sent,
        "score": sentence_similarities[idx]
    })

In [ ]:
print("\nEXPANDED SENTENCE CONTEXT:\n")

for r in expanded_results:
    print("Score:", r["score"])
    print("Prev:", r["previous"])
    print("MAIN:", r["sentence"])
    print("Next:", r["next"])
    print("-" * 60)


EXPANDED SENTENCE CONTEXT:

Score: 0.68593365
Prev: Reinforcement learning has been successful in applications as diverse as autonomous helicopter ﬂight, robot legged locomotion, cell-phone network routing, marketing strategy selection, factory control, and eﬃcient web-page indexing.
MAIN: Our study of reinforcement learning will begin with a deﬁnition of the Markov decision processes (MDP), which provides the formalism in which RL problems are usually posed.
Next: 187188 15.1 Markov decision processes A Markov decision process is a tuple ( S, A, {Psa}, γ, R), where: • S is a set of states.
------------------------------------------------------------
Score: 0.650049
Prev: Our study of reinforcement learning will begin with a deﬁnition of the Markov decision processes (MDP), which provides the formalism in which RL problems are usually posed.
MAIN: 187188 15.1 Markov decision processes A Markov decision process is a tuple ( S, A, {Psa}, γ, R), where: • S is a set of states.
Next: (For 